> **Cópia pública saneada.** Os dados de entrada não acompanham este repositório. Leia `docs/reprodutibilidade.md` e `docs/privacidade_e_dados.md` antes da execução. Notebooks de coleta dependem de rede; notebooks de tratamento escrevem somente em `data/`, que é ignorada pelo Git.

# Financiamento climático e economia verde no BNDES

## Resumo executivo

Este notebook mensura a participação do **Verde estrito** no total do BNDES. O indicador principal utiliza desembolsos mensais, enquanto as contratações são apresentadas como indicador complementar e não são somadas aos desembolsos.

Na execução mais recente, a participação real acumulada do Verde estrito nos desembolsos de 2002–2025 foi **1.15%**. A cobertura confirmada do pareamento representou **16.66%** do valor total; portanto, o resultado deve ser interpretado como estimativa conservadora sempre que a cobertura permanecer abaixo de 95%.

## 1. Metodologia e parâmetros

Os valores são corrigidos pelo IPCA mensal para preços constantes de junho de 2026. Desembolsos são deflacionados pelo mês da liberação e contratações pelo mês da contratação. Dezembro é um mês regular: o total anual corresponde à soma de janeiro a dezembro, e a média mensal é calculada separadamente.

A classificação principal contém apenas `Verde estrito` e `Demais operações`. Nenhum registro é excluído do denominador. Categorias históricas permanecem disponíveis apenas para auditoria.

### Preparação do ambiente

O bloco seguinte carrega o módulo reproduzível do projeto e executa toda a cadeia analítica: auditoria das fontes, classificação, IPCA, bases derivadas, resultados, Excel, figuras e testes de aceite.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bndes_financiamento_verde.analise_senior import executar_pipeline

artefatos = executar_pipeline()
artefatos.resumo

## 2. Auditoria das bases

Esta etapa confirma cobertura temporal, unidade de análise, quantidade de contratos e presença de duplicatas exatas. Os dois painéis permanecem independentes: liberações mensais no painel de desembolsos e registros/subcréditos no painel de contratações.

In [ ]:
display(artefatos.resultados["inventario_fontes"])
display(artefatos.resultados["qualidade_bases"])

## 3. Classificação binária e concordância histórica

A dimensão de políticas reconcilia obrigatoriamente 30 registros de Verde estrito e 212 de Demais operações. O numerador aceita apenas correspondências confirmadas; regras amplas produzem candidatos de revisão, mas não promovem registros ao conjunto verde.

In [ ]:
display(artefatos.classificacao["classificacao_analise"].value_counts().rename_axis("classificacao").to_frame("registros"))
display(artefatos.resultados["cobertura_classificacao"].head(20))

## 4. Deflação pelo IPCA

O fator mensal é a razão entre o número-índice do IPCA de junho de 2026 e o número-índice do mês de referência. O fator do mês-base deve ser exatamente igual a um, e todos os meses das duas bases precisam encontrar correspondência.

In [ ]:
display(artefatos.ipca.tail(8))
artefatos.ipca.loc[artefatos.ipca["data_referencia"].eq(pd.Timestamp("2026-06-01")), ["data_referencia", "ipca_mes", "fator_ipca_jun2026"]]

## 5. Painel de desembolsos

Os totais anuais são somas dos fluxos mensais. A participação verde utiliza numerador e denominador expressos nos mesmos preços constantes. O ano de 2026 é identificado como YTD e não entra nas comparações anuais principais.

In [ ]:
display(artefatos.resultados["desembolsos_anuais"])
display(Image(filename=str(ROOT / "results" / "figures" / "desembolsos_total_verde_reais.png")))
display(Image(filename=str(ROOT / "results" / "figures" / "participacao_verde_anual.png")))

## 6. Diagnóstico mensal e dezembro

A razão entre o desembolso de cada mês e a média mensal do respectivo ano permite verificar sazonalidade. Dezembro não é tratado como valor acumulado anual.

In [ ]:
display(artefatos.resultados["sazonalidade_mensal"].query("mes == 12").tail(10))
display(Image(filename=str(ROOT / "results" / "figures" / "diagnostico_sazonalidade_dezembro.png")))

## 7. Painel complementar de contratações

As operações automáticas são mantidas como registros operacionais. Nas não automáticas, os valores são somados por subcrédito, mas a contagem contratual utiliza o identificador do contrato. A tabela de robustez compara a regra principal com a remoção de duplicatas exatas.

In [ ]:
display(artefatos.resultados["contratacoes_anuais"])
display(artefatos.resultados["estatisticas_contratacoes"])
display(artefatos.resultados["robustez"])

## 8. Composição temática, setorial e territorial

A leitura segue a hierarquia solicitada: bloco temático, instrumento, linha, modalidade, setor e território. Para UFs e municípios, o ranking usa somente geografia válida e apresenta a parcela residual não territorializável em tabela própria.

In [ ]:
display(artefatos.resultados["cobertura_territorial"])
display(Image(filename=str(ROOT / "results" / "figures" / "composicao_bloco_tematico.png")))

## 9. Top 10

Os rankings são ordenados pelo valor real, com desempate pela quantidade de registros. São gerados para o período acumulado 2002–2025, para 2025 e para 2026 YTD.

In [ ]:
for chave in ["top10_instrumentos", "top10_linhas", "top10_setores", "top10_uf", "top10_municipios"]:
    print(f"\n{chave}")
    display(artefatos.resultados[chave].query("universo == 'Desembolsos' and recorte == '2002–2025'"))

## 10. Robustez, reconciliações e limites

Os testes verificam a reconciliação entre Verde estrito, Demais operações e Total BNDES; cobertura integral do IPCA; número correto de contratos; anos completos; ausência de categorias territoriais inválidas nos rankings; e separação conceitual entre desembolsos e contratações.

Como a dimensão de políticas não contém vigência histórica, o pareamento confirmado deve ser interpretado com cautela. Cobertura inferior a 95% impede apresentar o resultado como mensuração definitiva do universo ambiental do banco.

In [ ]:
pd.Series(artefatos.validacao["testes"], name="aprovado").to_frame()

## 11. Conclusão

O notebook entrega uma estimativa conservadora e auditável da participação do Verde estrito, sem excluir operações do denominador e sem misturar fluxos desembolsados com compromissos contratados. A planilha final, os parquets derivados e as figuras permitem replicar e aprofundar a análise.